In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

class MHA(nn.Module): # multi-head attention
    def __init__(self, num_head, dim):
        super().__init__()
        self.qkv_proj = nn.Linear(dim, 3*dim)
        self.dk = dim // num_head
        self.num_head = num_head

    def forward(self, x, mask, kv_cache=None): #kv_cache
        B, L, D = x.size() # batch_size, length, dimension
        qkv_proj = self.qkv_proj(x) # B, L, 3D
        q, k, v = torch.chunk(qkv_proj, 3, dim=-1) # 3 * B, L, D

        q = q.view(B, L, self.num_head, self.dk).transpose(1, 2) # B, num_head, L, dk
        k = k.view(B, L, self.num_head, self.dk).transpose(1, 2)
        v = v.view(B, L, self.num_head, self.dk).transpose(1, 2)

        if kv_cache:
            old_k, old_v = kv_cache
            k = torch.concat([old_k, k], dim=-1)
            v = torch.concat([old_v, v], dim=-1)
            new_kv_cache = (k, v)


        scores = torch.matmul(q, k.transpose(1, 2)) / torch.sqrt(self.dk)# (B, num_head, L, dk) x (B, num_head, dk, L) => (B, num_head, L, L)
        # casual mask
        if mask:
            scores.masked_fill(mask == 0, float('-inf'))

        scores = F.softmax(scores, dim=-1)
        output = torch.maxmul(scores, v) # (B, num_head, L, dk)
        output.transpose(1, 2).contiguous().view(B, L, D) # (B, L, D)
        return self.output_proj(output)


